# Swarm Pattern: Practical Demo

**No central coordinator.** Each agent decides, on its own, whether to answer or to **hand off** to another agent — by calling a real `handoff_to_X` tool, the same way our other notebook's agent calls `web_search`. The full conversation history passes directly to whichever agent takes over next; nobody starts fresh.

**Three peer agents:**
- **Billing agent:** knows the real billing policy. Can hand off to research or calculator.
- **Research agent:** has a real, live web-search tool. Can hand off to billing.
- **Calculator agent:** does real refund arithmetic (an LLM should not be trusted to compute this itself). Can hand off to billing.

**Entry point:** every conversation starts at the **billing** agent (a swarm still needs *some* starting point).

**What we will do**
1. Setup
2. The three real tools (billing has none of its own — just handoff tools)
3. The three agents, each with its own system prompt and tool set
4. Wire them into a graph: agents hand off to each other, with a loop guard
5. Run four real queries, from "no handoff needed" up to a forced multi-hop chain and a loop-risk case
6. Compare against the supervisor pattern

## 1. Setup

In [ ]:
# Bring the api_key from secrets into a variable
from google.colab import userdata

my_api_key = userdata.get('agentic_ai')

In [ ]:
pip install langchain-groq langgraph langchain-community ddgs -q # q for silent installation

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
from langchain_groq import ChatGroq

base_llm = ChatGroq(
    model = 'openai/gpt-oss-20b',
    temperature = 0, # Creativity level of the model
    api_key = my_api_key
)

## 2. The real tools

### 2.1 A real web search tool (for the research agent)

In [ ]:
from langchain_community.tools import DuckDuckGoSearchRun
from ddgs.exceptions import DDGSException

search = DuckDuckGoSearchRun()

def web_search(query: str) -> str:
    """Search the live web for current, up-to-date information."""
    try:
        return search.run(query)
    except DDGSException as e:
        return f'(Web search unavailable right now - {e}. No search results to go on.)'

/tmp/ipykernel_480/2917877972.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


### 2.2 A real refund calculator (for the calculator agent)

An LLM should not be trusted to do exact arithmetic. This is a real Python function, not an LLM guess.

In [ ]:
def calculate_refund(monthly_cost: float, months_remaining: int) -> float:
    """Calculate the refund for a yearly plan: unused months, minus a 10% processing fee."""
    return round(monthly_cost * months_remaining * 0.9, 2)

### 2.3 Handoff tools

A handoff tool does no real work; calling it IS the hand-off. Its return text becomes part of the shared conversation, so the next agent sees why it was called in.

In [ ]:
def handoff_to_billing(reason: str) -> str:
    """Hand off the conversation to the BILLING agent, for billing policy or refund questions."""
    return f"Transferring to the billing agent. Reason: {reason}"

def handoff_to_research(reason: str) -> str:
    """Hand off the conversation to the RESEARCH agent, for current/live information from the web."""
    return f"Transferring to the research agent. Reason: {reason}"

def handoff_to_calculator(reason: str) -> str:
    """Hand off the conversation to the CALCULATOR agent, to compute an exact refund amount."""
    return f"Transferring to the calculator agent. Reason: {reason}"

In [ ]:
# Billing policy (the billing agent's only source of truth)
billing_policy = """ShopSmart Billing Policy:
- Monthly plans: cancel anytime, no refund for the current month.
- Yearly plans: cancel anytime, refund the unused months, minus a 10% processing fee.
- Refunds are processed within 5-7 business days to the original payment method."""

## 3. The three agents

Each agent is bound to its **own** tools only. A billing agent that was never given `web_search` genuinely cannot search the web itself — it can only hand off.

In [ ]:
billing_tools    = [handoff_to_research, handoff_to_calculator]
research_tools   = [web_search, handoff_to_billing]
calculator_tools = [calculate_refund, handoff_to_billing]

llm_billing    = base_llm.bind_tools(billing_tools)
llm_research   = base_llm.bind_tools(research_tools)
llm_calculator = base_llm.bind_tools(calculator_tools)

In [ ]:
agent_prompts = {
    'billing': f"""You are the billing agent for ShopSmart.
Policy:
{billing_policy}
If the user asks about anything needing live web information, hand off to research.
If the user needs an exact refund number computed, hand off to calculator.""",

    'research': """You are the research agent for ShopSmart. You answer using live web search.
If the question is about billing policy or refunds, hand off to billing.""",

    'calculator': """You are the calculator agent for ShopSmart. You compute exact refund amounts.
Once you have computed the number, hand off to billing so it can give the final answer to the user.""",
}

In [ ]:
def make_agent_node(name, llm_with_tools):
    def node(state):
        print(f"\n--- {name.upper()} AGENT ---")
        messages = [('system', agent_prompts[name])] + state['messages']
        response = llm_with_tools.invoke(messages)
        if response.tool_calls:
            for call in response.tool_calls:
                print(f"  -> calls: {call['name']}({call['args']})")
        else:
            print("  -> answers directly (no handoff)")
        return {'messages': [response], 'current_agent': name}
    return node

## 4. Wiring the agents together

After any agent speaks: if it called a tool, run it. If that tool was a **handoff**, route to the named agent. If it was a **real** tool (search / calculator), route back to the **same** agent that called it, so it can use the result. A simple count stops an infinite back-and-forth.

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import ToolMessage

class State(TypedDict):
    messages: Annotated[list, add_messages]
    current_agent: str
    handoffs: int

In [ ]:
billing_node    = make_agent_node('billing', llm_billing)
research_node   = make_agent_node('research', llm_research)
calculator_node = make_agent_node('calculator', llm_calculator)

# One shared tool node: it can execute any tool, any agent might call
all_tools = [web_search, calculate_refund, handoff_to_billing, handoff_to_research, handoff_to_calculator]
tool_node = ToolNode(all_tools)

In [ ]:
def has_tool_calls(state) -> Literal['tools', END]:
    return 'tools' if state['messages'][-1].tool_calls else END

In [ ]:
MAX_HANDOFFS = 4  # the loop guard

def after_tools(state):
    last_tool_result = state['messages'][-1]
    was_handoff = isinstance(last_tool_result, ToolMessage) and last_tool_result.name.startswith('handoff_to_')
    return {'handoffs': state['handoffs'] + 1} if was_handoff else {}

In [ ]:
def route_after_tools(state) -> Literal['billing', 'research', 'calculator', END]:
    last_tool_result = state['messages'][-1]

    if last_tool_result.name.startswith('handoff_to_'):
        if state['handoffs'] > MAX_HANDOFFS:
            print(f"  >>> LOOP GUARD: stopped after {state['handoffs']} handoffs <<<")
            return END
        return last_tool_result.name.removeprefix('handoff_to_')  # e.g. 'billing'

    return state['current_agent']  # a real tool (search / calculator): go back to the same agent

In [ ]:
builder = StateGraph(State)
builder.add_node('billing', billing_node)
builder.add_node('research', research_node)
builder.add_node('calculator', calculator_node)
builder.add_node('tools', tool_node)
builder.add_node('after_tools', after_tools)

builder.add_edge(START, 'billing')  # fixed entry point

for agent in ['billing', 'research', 'calculator']:
    builder.add_conditional_edges(agent, has_tool_calls)

builder.add_edge('tools', 'after_tools')
builder.add_conditional_edges('after_tools', route_after_tools)

swarm_graph = builder.compile()

## 5. Run four real queries

- **Q1 (no handoff needed):** billing should answer on its own.
- **Q2 (one handoff):** billing should recognise the live-information part isn't its job.
- **Q3 (multi-hop chain):** should need billing -> calculator -> billing -> research, or a similar real chain.
- **Q4 (loop risk):** an ambiguous request that could bounce between agents, to see the guard in action.

In [ ]:
queries = {
    'Q1 (no handoff)':  'What is your refund policy for a yearly plan?',
    'Q2 (one handoff)': ('If I cancel today, what refund do I get, and is there any current '
                          'OpenAI API outage I should know about?'),
    'Q3 (multi-hop)':   ('My yearly plan is $20/month and I have used 5 months so far. '
                          'What is my exact refund, and is there a current OpenAI outage I should know about?'),
    'Q4 (loop risk)':   'Is my refund amount affected by any current billing API outage you can look up?',
}

In [ ]:
def run_swarm(query):
    initial_state = {'messages': [('user', query)], 'current_agent': 'billing', 'handoffs': 0}
    final_state = swarm_graph.invoke(initial_state, {'recursion_limit': 30})

    path = [m.name.removeprefix('handoff_to_') for m in final_state['messages']
            if isinstance(m, ToolMessage) and m.name.startswith('handoff_to_')]

    return final_state['messages'][-1].content, path, final_state['handoffs']

In [ ]:
for label, query in queries.items():
    print('=' * 70)
    print(label, ':', query)
    print('=' * 70)

    final_answer, path, handoffs = run_swarm(query)

    print(f"\nHANDOFF PATH: billing -> {' -> '.join(path) if path else '(none)'}")
    print(f"FINAL ANSWER: {final_answer}")
    print(f"Total handoffs: {handoffs}\n")

Q1 (no handoff) : What is your refund policy for a yearly plan?

--- BILLING AGENT ---
  -> answers directly (no handoff)

HANDOFF PATH: billing -> (none)
FINAL ANSWER: **ShopSmart Yearly Plan Refund Policy**

- **Cancel Anytime** – You can cancel your yearly plan whenever you wish.  
- **Refund** – We’ll refund the value of the unused months of your subscription.  
- **Processing Fee** – A 10 % processing fee is deducted from the refund amount.  
- **Timing** – The refund is processed within **5–7 business days** and will be credited back to the original payment method you used.  

If you’d like to see the exact refund amount for your specific situation, let me know the date you plan to cancel and I can calculate it for you.
Total handoffs: 0

Q2 (one handoff) : If I cancel today, what refund do I get, and is there any current OpenAI API outage I should know about?

--- BILLING AGENT ---
  -> calls: handoff_to_research({'reason': 'User wants to know about current OpenAI API outage'})


## 6. Comparison with the supervisor pattern

If you've run the supervisor notebook, compare them directly:

| | Supervisor | Swarm (this notebook) |
|---|---|---|
| Who decides the next step | One supervisor, every time | Whichever agent currently holds the conversation |
| What a worker/agent sees | A scoped sub-task the supervisor wrote | The full conversation history |
| Final answer | A separate synthesis call combines every worker's output | Whichever agent finishes last speaks directly to the user - there is no synthesis step |
| Risk | Bad decomposition (an unscoped or wrong sub-task) | Infinite handoff loops (needs a guard, as built above) |

**Why this matters in practice (from our notes):** swarm suits messy, drifting conversations (a billing question turns into a technical one) because each agent just says "this isn't mine, I'll transfer you" - no central router has to recognise every transition. The cost is that nothing is coordinating the whole conversation, which is exactly why a **loop guard** (`MAX_HANDOFFS`) is not optional - a real swarm system needs one.